# 02E — Four-forest leave-one-site-out transfer benchmark

**One selected method only**, read from `02D_selected_strategy.json` after the main within-site 3-fold model comparison completes. Train four models: withhold Baego, MOS (all three mosaics), Yangambi, or Yoko in turn; train on all remaining forest groups' partial annotations and exhaustive circles, without any labels from the target group. All 22 exhaustive circles are evaluated across their corresponding held-out forest test.

**Objective:** new-forest transferability, **not** optimizing the six existing orthomosaics. Full-circle fraction metrics primary, discrimination secondary. The V2 E1 illumination reference median is fit on only the three training forest groups each LOSO iteration; image-only luminance statistics at the target are permitted unsupervised and reported. No field plot use, no target reference calibration.

Checkpoints/scores: `Final/Outputs/V3/02E_LOSO/`. We intentionally do not overwrite 02C/02D outputs.


## 01 — Environment and load selected strategy

In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib scikit-learn
from pathlib import Path
import json, warnings, math, random, time
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                         Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None
CV_OUT=ROOT/'Outputs/V3/02D_spatial_cv'
BASE_OUT=ROOT/'Outputs/V3/02E_LOSO';BASE_OUT.mkdir(parents=True,exist_ok=True)
OUT=BASE_OUT
REF=ROOT/'Outputs/V3/01_reference_data'
A=json.loads((ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json').read_text())
mosaics={m['file']:m for m in A['mosaics']}
splits=pd.read_csv(REF/'01_cv_window_manifest.csv')
links=pd.read_csv(REF/'01_circle_spatial_folds.csv')
repairs=pd.read_csv(REF/'01_geometry_repairs.csv')
FOLD=0;SEED=20261016;CROP=256;EPOCHS=25;BATCH=8
TARGET_SITES=['Baego','MOS','Yangambi','Yoko']
SELECTED=json.loads((CV_OUT/'02D_selected_strategy.json').read_text())
SELECTED_MODEL=SELECTED['model'];SELECTED_TREATMENT=SELECTED['illumination']
assert SELECTED_MODEL in ['M1','M2']
assert SELECTED_TREATMENT in ['native','augment','v2_E1_site']
print('LOCKED STRATEGY',SELECTED_MODEL,SELECTED_TREATMENT)
MAX_PARTIAL_PER_MOSAIC=250
MAX_EXHAUSTIVE_PER_SITE=260
HOLDOUT_COUNTS={'Baego':1,'MOS':2,'Yangambi':2,'Yoko':2}
assert torch.cuda.is_available(),'Select Runtime > Change runtime type > GPU for the pilot.'
device='cuda';torch.manual_seed(SEED);np.random.seed(SEED);random.seed(SEED)
print('GPU',torch.cuda.get_device_name(0),'| fold',FOLD)
print('Train/val candidates:',splits[splits.cv_fold==FOLD].split.value_counts().to_dict())
print('Files:',ROOT,'| output:',OUT)





## 02 — Load masks and U-Net architecture

In [ ]:
def pieces(g):
    if g.geom_type in ('Polygon','MultiPolygon'):return [g]
    if hasattr(g,'geoms'):return [p for q in g.geoms for p in pieces(q)]
    return []
def polygons(path):
    src=gpd.read_file(path);inds=[];shapes=[]
    for i,g in enumerate(src.geometry):
        if g is None or g.is_empty:continue
        if not g.is_valid:
            try:
                fixed=unary_union(pieces(make_valid(g)))
                change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
                if change>.05 or not fixed.is_valid:continue
                g=fixed
            except Exception:continue
        inds.append(i);shapes.append(g)
    x=src.iloc[inds].copy().reset_index(drop=True)
    x.geometry=shapes
    x['original_row']=inds
    return x
roi=polygons(A['roi'])
circles=polygons(A['exhaustive_annotations']['complete_roi_boundaries'])
positive=polygons(A['exhaustive_annotations']['positive_polygons'])
circle_ids=set(links.circle_id.astype(int))
circles=circles[circles.original_row.isin(circle_ids)].copy()
tree={}
for site,path in A['shared_annotations'].items():
    t=polygons(path)
    cls=pd.to_numeric(t.Class_ID,errors='coerce')
    t['class01']=np.where(cls.eq(1),1,np.where(cls.notna(),0,-1))
    tree[site]=t
geo={}
for name,m in mosaics.items():
    with rasterio.open(m['path']) as src:
        c=circles.to_crs(src.crs).copy()
        c['fold']=c.original_row.map(links.set_index('circle_id').fold)
        c['owner']=c.original_row.map(links.set_index('circle_id').mosaic)
        geo[name]=dict(site=m['site'],path=m['path'],transform=src.transform,
            crs=src.crs,roi=roi.to_crs(src.crs),
            circles=c,positive=positive.to_crs(src.crs),trees=tree[m['site']].to_crs(src.crs))



In [ ]:
## 03 — Fold-specific masks and candidate windows

## 03 — LOSO reference masks: exclude all target-forest annotations

In [ ]:
## 04 — V2 E1 normalization: unsupervised fixed site luminance gains

Recovered from `V2/03_UAV_Illumination_Harmonization.ipynb`: site median luminance, reference median of the four training forest groups, global gain bounds **(0.85, 1.15)**, valid-pixel mask and unchanged original TIFFs. Samples are dispersed across each native mosaic; MOS is grouped as one forest. Site statistics use imagery only, not reference labels. The V2 local E2/E3 methods are deliberately excluded from this first controlled comparison
CANDIDATES_FULL=candidate.copy()
def configure_loso(held_site):
    global FOLD,DEV,HOLDOUT,BLOCKED,split_df,candidate,TARGET_SITE
    TARGET_SITE=held_site
    FOLD=-1
    HOLDOUT=set(links.loc[links.site==held_site,'circle_id'].astype(int))
    DEV=set(links.circle_id.astype(int))-HOLDOUT
    candidate=CANDIDATES_FULL[CANDIDATES_FULL.site!=held_site].copy()
    split_df=links[['circle_id','site','mosaic','cover_pct']].copy()
    split_df['split']=np.where(split_df.circle_id.isin(HOLDOUT),'holdout','development')
    BLOCKED={}
    for name,d in geo.items():
        cs=circles.to_crs(d['crs'])
        d['dev_circles']=cs[cs.original_row.isin(DEV)]
        d['holdout_circles']=cs[cs.original_row.isin(HOLDOUT)]
        held_here=set(links[(links.site==d['site'])&links.circle_id.isin(HOLDOUT)].circle_id.astype(int))
        BLOCKED[name]=[g.buffer(15.) for g in cs[cs.original_row.isin(held_here)].geometry]
    print('LOSO target',held_site,'target circles',sorted(HOLDOUT),
          'training candidate sites',candidate.site.unique().tolist(),flush=True)
    assert held_site not in candidate.site.unique()
    assert set(candidate.site.unique())==set(['Baego','MOS','Yangambi','Yoko'])-{held_site}


## 04 — E1 gains with strictly training-site reference, or native/augmentation treatment

In [ ]:
## 05 — Train M1/M2: native, augmented or V2 E1

Identical label policy and training epochs. Augmented treatment uses stochastic brightness, contrast and gamma on the RGB input only; target and supervision masks are unchanged. Model checkpoints written per fold/treatment to Drive
MOSAIC_LUM={name:sample_mosaic_luminance(name) for name in geo}
SITE_LUM={}
for site in ['Baego','MOS','Yangambi','Yoko']:
    values=[v for name,(v,n) in MOSAIC_LUM.items() if geo[name]['site']==site]
    SITE_LUM[site]=float(np.median(values))
SITE_GAIN={}
def configure_site_illumination(held_site):
    global SITE_GAIN,REF_LUM
    other=[s for s in SITE_LUM if s!=held_site]
    REF_LUM=float(np.median([SITE_LUM[s] for s in other]))
    SITE_GAIN={s:float(np.clip(REF_LUM/SITE_LUM[s],.85,1.15)) for s in SITE_LUM}
    print('LOSO reference luminance (TRAINING SITES ONLY):',REF_LUM,
          '| target unsupervised gain',SITE_GAIN[held_site],flush=True)
    return dict(heldout_site=held_site,training_sites=other,
                site_gains=SITE_GAIN,train_reference=REF_LUM)
.

## 05 — Selected model training implementation

In [ ]:
## 06 — Full-circle predicted fractions; all pixels inside held-out circles

Avoid the previous fixed 7-circle split. Validate every exhaustive circle exactly once per variant; RGB remains at native resolution and no background outside the circle is assumed labelled.

## 06 — Full-circle LOSO evaluation

In [ ]:
## 07 — Run/resume 18 fits and held-out evaluation

A complete (fold, treatment, model) pair has both a model checkpoint and a circle-score CSV. Completed runs are not retrained. We run M1 and M2, but M3 calibration is deliberately not part of the main benchmark after its negative result in 02C.

## 07 — Run/resume four strict site holdouts

In [ ]:
import gc
RGB_TREATMENT=SELECTED_TREATMENT
for site in TARGET_SITES:
    configure_loso(site)
    OUT=BASE_OUT/site
    OUT.mkdir(parents=True,exist_ok=True)
    gains=configure_site_illumination(site)
    (OUT/'02E_illumination_contract.json').write_text(json.dumps(gains,indent=2))
    score_file=OUT/f'{SELECTED_MODEL}_circle_raw.csv'
    checkpoint=OUT/f'{SELECTED_MODEL}_model.pt'
    if score_file.exists() and checkpoint.exists():
        print('SKIP completed LOSO:',site,flush=True)
        continue
    model=fit_network(SELECTED_MODEL)
    result,_=run_all_circles(model,SELECTED_MODEL)
    assert result.circle_id.nunique()==len(HOLDOUT),(site,len(result),len(HOLDOUT))
    assert set(result.site)=={site}
    print('LOSO completed:',site,'circles',len(result),'fraction MAE pp',
          round(float(result.bias_pp.abs().mean()),3),flush=True)
    del model;torch.cuda.empty_cache();gc.collect()
print('Four-site LOSO runs completed or resumed.')


## 08 — Four-site transferability summary and observed/predicted plot

In [ ]:
frames=[]
for site in TARGET_SITES:
    file=BASE_OUT/site/f'{SELECTED_MODEL}_circle_raw.csv'
    assert file.exists(),f'Run Section 07 first: {file}'
    frames.append(pd.read_csv(file))
LOSO=pd.concat(frames,ignore_index=True)
assert LOSO.circle_id.nunique()==22
def metrics(q):
    y=q.observed_fraction.to_numpy(float);p=q.predicted_fraction.to_numpy(float)
    r=linregress(y,p) if len(q)>=3 and len(np.unique(y))>1 else None
    return pd.Series(dict(n_circles=len(q),R2=r2_score(y,p) if len(q)>=3 else np.nan,
        MAE_pp=mean_absolute_error(y,p),RMSE_pp=np.sqrt(mean_squared_error(y,p)),
        mean_bias_pp=float(np.mean(p-y)),slope=r.slope if r else np.nan,
        intercept_pp=r.intercept if r else np.nan,brier=q.brier.mean(),
        AUC=q.auc_sample.mean(),AP=q.ap_sample.mean()))
BY_SITE=LOSO.groupby('site').apply(metrics,include_groups=False).reset_index()
OVERALL=metrics(LOSO).to_frame().T
display(BY_SITE.round(3));display(OVERALL.round(3))
LOSO.to_csv(BASE_OUT/'02E_all_site_holdouts.csv',index=False)
BY_SITE.to_csv(BASE_OUT/'02E_site_metrics.csv',index=False)
OVERALL.to_csv(BASE_OUT/'02E_overall_metrics.csv',index=False)
fig,ax=plt.subplots(figsize=(7,7))
ax.plot([0,100],[0,100],ls='--',color='gray')
colors=dict(zip(TARGET_SITES,['#c27e24','#357ebd','#468c50','#944b99']))
for site,g in LOSO.groupby('site'):
    ax.scatter(g.observed_fraction,g.predicted_fraction,label=site,color=colors[site],s=46)
ax.set(xlim=(0,100),ylim=(0,100),xlabel='Observed canopy fraction (%)',
       ylabel='Predicted canopy fraction (%)',
       title='Four-site LOSO: entirely unseen forest groups')
ax.legend();ax.grid(alpha=.2)
fig.savefig(BASE_OUT/'02E_LOSO_observed_predicted.png',dpi=180)
display(fig);plt.close(fig)
print('This is transferability evidence; choose the production approach using 02D spatial CV.')
